# TP1 ► Mise en place d’un environnement collaboratif pour l’entraˆınement d’un mod`ele d’IA
# Objectifs du TP
L’objectif de ce TP est de mettre en place un environnement collaboratif permettant a plusieurs utilisateurs de travailler sur un mˆeme projet d’intelligence artificielle, en utilisant Google Drive pour partager les fichiers et Google Colab pour l’entraˆınement des modeles. A l’issue de ce TP, l’´etudiant sera capable de :
- Creer et organiser un espace de travail sur Google Drive.
- Partager cet espace avec plusieurs utilisateurs.
- Connecter Google Colab au dossier partag´e.
- Organiser les fichiers d’un projet de Deep Learning.
- Sauvegarder p´eriodiquement les checkpoints d’un modele.
- Reprendre un entraˆınement interrompu `a partir du dernier checkpoint disponible.
- Collaborer avec plusieurs utilisateurs en r´epartissant les sessions d’entraˆınement GPU. 

# 2. Contexte et probl´ematique

L’entraˆınement des modeles de Deep Learning n´ecessite souvent des ressources GPU importantes et des dur´ees de calcul consid´erables. Les plateformes telles que Google Colab imposent des limites de dur´ee aux sessions d’ex´ecution et a la disponibilit´e des GPU.
Dans un contexte collaboratif, plusieurs ´etudiants peuvent se partager l’entraˆınement d’un mˆeme modele. Lorsqu’un utilisateur atteint la limite de sa session, un autre utilisateur doit pouvoir reprendre l’entraˆınement sans recommencer les epochs d´eja effectu´ees.
Ce TP consiste a mettre en place une solution bas´ee sur Google Drive et Google Colab permettant de sauvegarder l’´etat d’un mod`ele et de reprendre son entraˆınement depuis une autre session.

# 3. Partie 1 : Cr´eation du d´epˆot Google Drive

Cr´eer et organiser un espace de stockage d´edi´e au projet d’intelligence artificielle.

1. Se connecter `a son compte Google Drive.
9. Cr´eer un dossier nomm´e Projet IA Collaboratif.
9. Cr´eer l’arborescence suivante :
```
.
├── checkpoints/
│   ├── best.pth
│   └── latest.pth
├── datasets/
│   ├── test
│   ├── train
│   └── val
├── logs/
├── notebooks/
│   ├── TP1.ipynb
│   └── TP3.ipynb
├── results/
└── src/
    ├── evaluate.py
    ├── model.py
    └── train.py
```
9. Identifier le rˆole de chaque dossier.
9. D´eposer un petit jeu de donn´ees de test dans le dossier datasets.
9. Cr´eer un notebook d’entraˆınement dans le dossier notebooks.

# 4. Partie 2 : Partage du d´epˆot

Permettre a plusieurs utilisateurs d’acc´eder au mˆeme espace de travail.

1. Ouvrir les param`etres de partage du dossier Projet IA Collaboratif.
2. Ajouter les adresses Gmail des autres ´etudiants.
3. Attribuer le rˆole ´Editeur aux utilisateurs qui doivent modifier les fichiers et sauvegarder les checkpoints. 
4. V´erifier que chaque ´etudiant peut acc´eder au dossier depuis son propre compte.
5. Tester la cr´eation et la lecture d’un fichier partag´e.

Le partage d’un dossier ne garantit pas la coh´erence des ´ecritures simultan´ees. Un seul utilisateur doit ˆetre responsable de l’entraˆınement et de la sauvegarde d’un checkpoint donn´e `a un instant donn´e.

# 5. Partie 3 : Connexion de Google Colab`a Google Drive

Monter Google Drive dans Google Colab et acc´eder au projet partag´e.

1. Ouvrir Google Colab avec son compte Google.
2. Cr´eer un nouveau notebook Python.
3. Monter Google Drive :

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

4. D´efinir le chemin du projet :

In [ ]:
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/",
    "Projet_IA_Collaboratif "
)

print(os.listdir(PROJECT_DIR))

5. V´erifier l’acces aux dossiers datasets, src et checkpoints.
6. Verifier que le notebook peut lire et ´ecrire dans le dossier partag´e.

Le chemin exact du dossier partag´e d´epend de la maniere dont il est ajout´e a Google Drive. Si le dossier est accessible par un raccourci dans Mon Drive, utiliser le chemin correspondant a ce raccourci

# 6. Partie 4 : Entraˆınement et sauvegarde du mod`ele

Entraˆıner un modele et sauvegarder r´egulierement son ´etat afin de permettre la reprise de l’entraˆınement.

## 6.1. Configuration de l’environnement

1. S´electionner un GPU dans Google Colab `a partir du menu Ex´ecution ¿ Modifier le type d’ex´ecution.
2. Installer ou v´erifier les biblioth`eques n´ecessaires

In [4]:
import torch

device = torch . device (
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(" Device :", device)

 Device : cpu


## 6.2. Sauvegarde du checkpoint

Dans cette partie, on considere un modele PyTorch. Le checkpoint contient les ´el´ements suivants :

In [3]:
def save_checkpoint(model, epoch, best_metric, path):
    checkpoint = {
        "epoch": epoch,
        "best_metric": best_metric,
        "model_state_dict": model.state_dict()
    }

    torch.save(checkpoint, path)

## 6.3. Sauvegarde periodique

Le checkpoint peut ˆetre sauvegard´e `a intervalles r´eguliers, par exemple toutes les cinq epochs.

In [ ]:
checkpoint_dir = os.path.join(PROJECT_DIR, " checkpoints ")
os.makedirs(checkpoint_dir, exist_ok = True)

latest_path = os.path.join(checkpoint_dir, " latest.pth ")
best_path = os.path.join(checkpoint_dir, "best.pth")

best_metric = 0.0

for epoch in range(num_epochs):
    train_one_epoch(model, train_loader, optimizer)
    val_metric = validate (model, val_loader)
    
    if val_metric > best_metric:
        best_metric = val_metric
        save_checkpoint(model, epoch, best_metric, best_path)
        
    save_checkpoint(model ,epoch, best_metric, latest_path)
    print(f"Epoch { epoch + 1}/{num_epochs} "
          f" - Val metric : {val_metric :.4f} "
          f" - Best : {best_metric :.4f} "
     )

Dans cet exemple, `latest.pth` contient le dernier ´etat sauvegard´e, tandis que `best.pth` contient le mod`ele ayant obtenu la meilleure m´etrique de validation.

# 7. Partie 5 : Reprise de l’entraınement

Reprendre l’entraˆınement `a partir du dernier checkpoint sauvegard´e par un autre utilisateur. Lorsqu’un utilisateur atteint la limite de sa session GPU, il doit sauvegarder son dernier checkpoint. Un autre utilisateur peut alors reprendre l’entraˆınement depuis ce checkpoint.

## 7.1. Chargement du checkpoint

Le nouvel utilisateur doit initialiser la mˆeme architecture de modele, puis charger les parametres sauvegard´es.

In [ ]:
checkpoint = torch.load(
    latest_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_metric = checkpoint["best_metric"]

print("Reprise à partir de l'epoch :", start_epoch)
print("Meilleure métrique :", best_metric)

## 7.2. Poursuite de l’entraınement

Apres le chargement du checkpoint, l’entraˆınement peut ˆetre poursuivi a partir de la prochaine epoch.

In [ ]:
for epoch in range(start_epoch, num_epochs):
    train_one_epoch(
        model,
        train_loader,
        optimizer,
    )

    val_metric = validate(
        model,
        val_loader,
    )

    
    if val_metric > best_metric:
        best_metric = val_metric
    
        save_checkpoint(
            model,
            epoch,
            best_metric,
            best_path,
        )
    
    save_checkpoint(
        model,
        epoch,
        best_metric,
        latest_path,
    )
    
    print(
        f"epoch {epoch + 1}/{num_epochs} "
        f"- val metric : {val_metric:.4f} "
        f"- best : {best_metric:.4f}"
    )

Cette version sauvegarde les poids du modele, le num´ero de l’epoch et la meilleure m´etrique. Pour une reprise strictement ´equivalente l’entraˆınement initial, il faut ´egalement sauvegarder et restaurer l’´etat de l’optimiseur, du scheduler et, le cas ´ech´eant, du GradScaler et des g´en´erateurs al´eatoires.

# 8. Partie 6 : Validation du travail collaboratif

Verifier le bon fonctionnement du partage et de la reprise de l’entraˆınement entre deux utilisateurs. Le travail doit ˆetre r´ealis´e par deux ´etudiants:

1. L’´etudiant A lance l’entraˆınement du mod`ele pendant cinq epochs.
2. Il v´erifie la pr´esence du fichier `latest.pth` dans Google Drive.
3. Il arrˆete sa session GPU apr`es avoir v´erifi´e la fin de la sauvegarde.
4. L’´etudiant B ouvre le notebook depuis son propre compte Google Colab.
5. Il monte le dossier partag´e et charge `latest.pth`.
6. Il reprend l’entraˆınement a partir de la sixieme epoch.
7. Il effectue cinq epochs suppl´ementaires.
8. Il v´erifie que le dernier checkpoint contient le num´ero de la derni`ere epoch r´ealis´ee.